# CODI seed 2 — Kaggle evaluation-only recovery

Use this notebook only when `codi_seed2` training reached step 96,405 but the saved dataset has no final evaluation. Attach the Kaggle dataset `jonraza15/codi-seed-2-resume-dataset`, enable a GPU and Internet, then use **Save Version → Save & Run All**.

This notebook never invokes the training runner. It verifies the experiment identity and final checkpoint, evaluates 200 examples per dataset (MultiArith has 180), retains only the final checkpoint, records a SHA-256 checksum, and uploads `jonraza15/codi-seed2-final-step96405`.

In [ ]:
EXPERIMENT = "codi_seed2"
EXPECTED_METHOD = "codi"
EXPECTED_SEED = 2
EXPECTED_STEP = 96405
EVAL_LIMIT = 200

REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "d917bef2cf396fe3b0453e6f86648f1a3948f528"
REPO_DIR = "/kaggle/working/latent-reasoning"
EXPORT_ROOT = "/kaggle/working/codi_seed2_final_export"
FINAL_DATASET_HANDLE = "jonraza15/codi-seed2-final-step96405"
UPLOAD_FINAL_DATASET = True


## 1. Pin the repository and verify the GPU

In [ ]:
import os, pathlib, shutil, subprocess, sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kagglehub>=1.0.2"],
    check=True,
)

os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == RUN_COMMIT, f"Commit mismatch: {commit} != {RUN_COMMIT}"

import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
print("Checked out:", commit)
print("Torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))


## 2. Locate, validate, and stage the final checkpoint

The attached input may contain both step 96,000 and step 96,405. Only step 96,405 is copied into the final export.

In [ ]:
from pathlib import Path
import json

input_checkpoints = [
    path
    for path in Path("/kaggle/input").rglob(f"step_{EXPECTED_STEP:08d}.pt")
    if path.parent.name == "checkpoints"
    and path.parent.parent.name == EXPERIMENT
]
assert len(input_checkpoints) == 1, (
    f"Expected exactly one {EXPERIMENT} step-{EXPECTED_STEP} checkpoint; "
    f"found {len(input_checkpoints)}: {input_checkpoints}"
)

input_checkpoint = input_checkpoints[0]
input_source = input_checkpoint.parents[1]
manifest_path = input_source / "run_manifest.json"
assert manifest_path.is_file(), f"Missing run manifest: {manifest_path}"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
resume = manifest.get("resume_config", {})
effective = manifest.get("effective_config", {})

assert resume.get("seed") == EXPECTED_SEED
assert resume.get("run_name") == EXPERIMENT
assert resume.get("task", {}).get("method") == EXPECTED_METHOD
assert effective.get("train", {}).get("total_steps") == EXPECTED_STEP
fingerprint = manifest.get("fingerprint")
assert isinstance(fingerprint, str) and len(fingerprint) == 64

export_root = Path(EXPORT_ROOT)
output_dir = export_root / "latent-reasoning/outputs/controls_and_seeds" / EXPERIMENT
if export_root.exists():
    shutil.rmtree(export_root)
(output_dir / "checkpoints").mkdir(parents=True)
staged_checkpoint = output_dir / "checkpoints" / input_checkpoint.name
shutil.copy2(input_checkpoint, staged_checkpoint)
shutil.copy2(manifest_path, output_dir / "run_manifest.json")

from scripts.colab_runner import validate_checkpoint_payload, validate_torch_checkpoint_archive
validate_torch_checkpoint_archive(staged_checkpoint)
validate_checkpoint_payload(staged_checkpoint, EXPECTED_STEP)
print("Input source:", input_source)
print("Staged checkpoint:", staged_checkpoint)
print("Manifest fingerprint:", fingerprint)


## 3. Evaluate only — no training

In [ ]:
from scripts.colab_control_runner import build_experiment_config
from src.eval.run_eval import evaluate

cfg = build_experiment_config(
    EXPERIMENT,
    output_dir=output_dir,
    max_seconds=1,
    keep_last=1,
)
results = evaluate(cfg, limit=EVAL_LIMIT)
print("Evaluation results:", results)


## 4. Verify evaluation and record an export audit

In [ ]:
import hashlib
from datetime import datetime, timezone

eval_dir = output_dir / "eval" / f"step_{EXPECTED_STEP:08d}"
summary_path = eval_dir / "summary.json"
assert summary_path.is_file(), "Final evaluation summary is missing"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary.get("checkpoint_step") == EXPECTED_STEP
assert summary.get("method") == EXPECTED_METHOD
assert set(summary.get("datasets", {})) == {"gsm8k", "svamp", "multiarith", "gsm_hard"}
for dataset in summary["datasets"]:
    assert (eval_dir / f"{dataset}.jsonl").is_file(), f"Missing predictions for {dataset}"

digest = hashlib.sha256()
with staged_checkpoint.open("rb") as handle:
    for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
        digest.update(block)
checkpoint_sha256 = digest.hexdigest()
(output_dir / "SHA256SUMS.txt").write_text(
    f"{checkpoint_sha256}  checkpoints/{staged_checkpoint.name}\n",
    encoding="utf-8",
)
audit = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": EXPERIMENT,
    "method": EXPECTED_METHOD,
    "seed": EXPECTED_SEED,
    "checkpoint_step": EXPECTED_STEP,
    "checkpoint_sha256": checkpoint_sha256,
    "manifest_fingerprint": fingerprint,
    "source_checkpoint": str(input_checkpoint),
    "source_commit": commit,
    "eval_limit": EVAL_LIMIT,
    "summary": summary,
}
(output_dir / "evaluation_export.json").write_text(
    json.dumps(audit, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, indent=2, sort_keys=True))
print("Checkpoint SHA256:", checkpoint_sha256)
print("Verified export:", export_root)


## 5. Upload the complete final dataset

Kaggle notebooks authenticate KaggleHub automatically. The first run creates the dataset; a later rerun creates another version.

In [ ]:
if UPLOAD_FINAL_DATASET:
    import kagglehub
    kagglehub.dataset_upload(
        FINAL_DATASET_HANDLE,
        str(export_root),
        version_notes=(
            f"Complete {EXPERIMENT} step {EXPECTED_STEP} with "
            f"numeric exact-match evaluation limit {EVAL_LIMIT}"
        ),
    )
    print("Final dataset uploaded:", FINAL_DATASET_HANDLE)
else:
    print("Upload skipped; export remains at:", export_root)


## Handoff

After Kaggle reports that the dataset files are processed, download `jonraza15/codi-seed2-final-step96405` in Colab and run `scripts/import_kaggle_control.py --experiment codi_seed2`. The importer should report `state: complete` and `checkpoint_step: 96405`.